In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 8.7 MB/s eta 0:00:00


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import optuna
import shap
import time

In [ ]:
# Загрузка и подготовка данных
df = pd.read_csv("hypertension_dataset.csv")
df['Medication'] = df['Medication'].fillna('None')
bp_history_mapping = {'Normal': 0, 'Prehypertension': 1, 'Hypertension': 2}
df['BP_History'] = df['BP_History'].map(bp_history_mapping)
categorical_cols = ['Family_History', 'Exercise_Level', 'Smoking_Status', 'Medication']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
df['Has_Hypertension'] = df['Has_Hypertension'].map({'Yes': 1, 'No': 0})

X = df.drop('Has_Hypertension', axis=1)
y = df['Has_Hypertension']

# Масштабирование числовых признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)


# Optuna – это инструмент, предназначенный для автоматического подбора наилучших параметров для моделей машинного обучения.

Optuna выделяется использованием умных
алгоритмов, которые анализируют предыдущие попытки и сосредотачиваются
на наиболее перспективных областях параметров. По эффективности Optuna
превосходит Grid Search и Random Search. Фреймворк гибок в настройке
параметров, прост в использовании, предоставляет инструменты
визуализации,
поддерживает
параллелизацию
бесперспективные попытки обучения.

In [ ]:
def objective(trial):
    """Функция цели для Optuna, которая минимизирует ошибку (максимизирует точность)."""

    # Гиперпараметры для оптимизации
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20, log=True),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'random_state': 42
    }

    model = RandomForestClassifier(**params)
    # Используем кросс-валидацию для оценки качества
    score = cross_val_score(model, X_train, y_train, n_jobs=-1, cv=5, scoring='accuracy').mean()
    return score

# Запуск оптимизации
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

# Получение лучших результатов
best_params = study.best_params
print(f"Оптимальные гиперпараметры (Optuna): {best_params}")

# Обучение финальной модели
best_rf = RandomForestClassifier(**best_params, random_state=42)
best_rf.fit(X_train, y_train)

# Оценка качества на тестовой выборке
y_pred = best_rf.predict(X_test)
print("\nОтчет по классификации (Оптимизированный RF на исходных данных):")
print(classification_report(y_test, y_pred))


[I 2025-11-11 16:20:46,986] A new study created in memory with name: no-name-fe57e79d-3161-4381-b9c4-88f4d7acbc04


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-11-11 16:20:51,224] Trial 0 finished with value: 0.9222320338674909 and parameters: {'n_estimators': 217, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 3, 'criterion': 'gini'}. Best is trial 0 with value: 0.9222320338674909.
[I 2025-11-11 16:20:52,613] Trial 1 finished with value: 0.9582448120925642 and parameters: {'n_estimators': 123, 'max_depth': 7, 'min_samples_split': 15, 'min_samples_leaf': 7, 'criterion': 'entropy'}. Best is trial 1 with value: 0.9582448120925642.
[I 2025-11-11 16:20:54,066] Trial 2 finished with value: 0.9647118406357945 and parameters: {'n_estimators': 106, 'max_depth': 14, 'min_samples_split': 3, 'min_samples_leaf': 2, 'criterion': 'gini'}. Best is trial 2 with value: 0.9647118406357945.
[I 2025-11-11 16:20:54,912] Trial 3 finished with value: 0.9402332285795912 and parameters: {'n_estimators': 98, 'max_depth': 5, 'min_samples_split': 13, 'min_samples_leaf': 6, 'criterion': 'gini'}. Best is trial 2 with value: 0.9647118406357945.
[I 2025

In [ ]:
# Задание 2: Определение оптимального количества компонент LDA

# Теоретическое ограничение LDA для бинарной классификации (C=2)
max_lda_components = len(np.unique(y_train)) - 1
print(f"Максимально возможное количество компонент LDA: {max_lda_components}")

# Применяем LDA с 1 компонентой
lda = LDA(n_components=1)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

# Обучаем Random Forest на одномерных данных LDA
rf_lda = RandomForestClassifier(**best_params, random_state=42)
rf_lda.fit(X_train_lda, y_train)
y_pred_lda = rf_lda.predict(X_test_lda)

accuracy_lda = accuracy_score(y_test, y_pred_lda)

print(f"\nТочность RF на 1 компоненте LDA: {accuracy_lda:.4f}")
print("Отчет по классификации (RF на 1 компоненте LDA):")
print(classification_report(y_test, y_pred_lda))


Максимально возможное количество компонент LDA: 1

Точность RF на 1 компоненте LDA: 0.7919
Отчет по классификации (RF на 1 компоненте LDA):
              precision    recall  f1-score   support

           0       0.77      0.81      0.79       286
           1       0.82      0.77      0.79       310

    accuracy                           0.79       596
   macro avg       0.79      0.79      0.79       596
weighted avg       0.79      0.79      0.79       596



## Анализ:
Оптимизированная модель Random Forest (RF) на исходных данных демонстрирует высокие показатели точности, полноты и F1-меры. В частности, для класса 0 точность составляет 0.95, полнота — 0.97, а для класса 1 — 0.97 по точности и 0.95 по полноте. Общая точность модели составляет 96%, что свидетельствует о ее высокой эффективности.

В то же время, когда модель RF была применена к данным, преобразованным с помощью LDA (линейного дискриминантного анализа), результаты значительно ухудшились. Точность модели на 1 компоненте LDA составила 79%. Показатели precision и recall для классов также снизились, что указывает на потерю информации и ухудшение классификации.

## Вывод:
Использование оригинальных данных без предварительной обработки через LDA приводит к значительно лучшим результатам в классификации по сравнению с использованием одной компоненты LDA. Это может свидетельствовать о том, что в данном случае LDA не улучшает, а наоборот, ухудшает качество модели.